**ROBERTA**

In [ ]:
!nvidia-smi

Tue Jan 13 12:11:56 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 570.172.08             Driver Version: 570.172.08     CUDA Version: 12.8     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   35C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
import pandas as pd
import os
os.environ["TORCH_COMPILE"] = "0"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

!pip install -q pyarrow fastparquet

# Read Parquet file from Google Drive
AAPL_News_only= pd.read_parquet('/kaggle/input/aapl-news-only')

AAPL_News_only.head()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 26.7 MB/s eta 0:00:00a 0:00:01


,date,time,title,content,symbols
0,2012-05-04,00:00:00,Why Erie Indemnity's Shares Dropped,Is this meaningful? Or just another movement?,AAPL
1,2015-09-25,04:01:00,"Security Solutions by AMAG, Identiv, ImageWare...","NEW YORK, NY--(Marketwired - Sep 25, 2015) - I...",AAPL
2,2015-11-03,15:00:00,FIDO Alliance Announces 72 Certified Authentic...,"MOUNTAIN VIEW, CA--(Marketwired - Nov 3, 2015)...",AAPL
3,2016-02-19,15:05:00,Payment Data Systems Announces Apple Pay Suppo...,"SAN ANTONIO, Feb. 19, 2016 (GLOBE NEWSWIRE) ...",AAPL
4,2016-03-07,22:00:00,NTT DOCOMO Rolls Out FIDO Biometric Authentica...,"MOUNTAIN VIEW, CA--(Marketwired - Mar 7, 2016)...",AAPL


In [ ]:
AAPL_News_only.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19005 entries, 0 to 19004
Data columns (total 5 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   date     19005 non-null  object
 1   time     19005 non-null  object
 2   title    19005 non-null  object
 3   content  19005 non-null  object
 4   symbols  19005 non-null  object
dtypes: object(5)
memory usage: 742.5+ KB


In [ ]:
#REGEX
import pandas as pd
import re

def clean_text_for_sentiment(text):
    # Handle missing values safely
    if not isinstance(text, str):
        return ""

    # 1. Remove URLs (Links don't carry sentiment)
    text = re.sub(r"http\S+|www\S+", "", text)

    # 2. Remove non-ASCII characters
    # This removes emojis and weird encoding symbols while keeping text clean
    text = text.encode("ascii", "ignore").decode()

    # 3. Clean characters but PRESERVE sentiment-relevant punctuation
    # We keep: Letters, Numbers, Spaces, and . , ! ? ' -
    text = re.sub(r"[^A-Za-z0-9\s.,!?'-]", " ", text)

    # 4. Normalize whitespace (remove extra spaces and newlines)
    text = re.sub(r"\s+", " ", text)

    return text.strip()

# Apply the cleaning function to your dataframe
AAPL_News_only["content"] = AAPL_News_only["content"].apply(clean_text_for_sentiment)

# Verify the results
print("Cleaning complete. Preview of cleaned news:")
print(AAPL_News_only["content"].head())


Cleaning complete. Preview of cleaned news:
0        Is this meaningful? Or just another movement?
1    NEW YORK, NY-- Marketwired - Sep 25, 2015 - Id...
2    MOUNTAIN VIEW, CA-- Marketwired - Nov 3, 2015 ...
3    SAN ANTONIO, Feb. 19, 2016 GLOBE NEWSWIRE -- P...
4    MOUNTAIN VIEW, CA-- Marketwired - Mar 7, 2016 ...
Name: content, dtype: object


In [ ]:
AAPL_News_tokenized = AAPL_News_only.copy()
AAPL_News_tokenized.head()

,date,time,title,content,symbols
0,2012-05-04,00:00:00,Why Erie Indemnity's Shares Dropped,Is this meaningful? Or just another movement?,AAPL
1,2015-09-25,04:01:00,"Security Solutions by AMAG, Identiv, ImageWare...","NEW YORK, NY-- Marketwired - Sep 25, 2015 - Id...",AAPL
2,2015-11-03,15:00:00,FIDO Alliance Announces 72 Certified Authentic...,"MOUNTAIN VIEW, CA-- Marketwired - Nov 3, 2015 ...",AAPL
3,2016-02-19,15:05:00,Payment Data Systems Announces Apple Pay Suppo...,"SAN ANTONIO, Feb. 19, 2016 GLOBE NEWSWIRE -- P...",AAPL
4,2016-03-07,22:00:00,NTT DOCOMO Rolls Out FIDO Biometric Authentica...,"MOUNTAIN VIEW, CA-- Marketwired - Mar 7, 2016 ...",AAPL


In [ ]:
# Normalize tokens: lowercase, remove extra spaces
AAPL_News_tokenized["content"] = AAPL_News_tokenized["content"].str.lower().str.strip()
AAPL_News_tokenized.head()

,date,time,title,content,symbols
0,2012-05-04,00:00:00,Why Erie Indemnity's Shares Dropped,is this meaningful? or just another movement?,AAPL
1,2015-09-25,04:01:00,"Security Solutions by AMAG, Identiv, ImageWare...","new york, ny-- marketwired - sep 25, 2015 - id...",AAPL
2,2015-11-03,15:00:00,FIDO Alliance Announces 72 Certified Authentic...,"mountain view, ca-- marketwired - nov 3, 2015 ...",AAPL
3,2016-02-19,15:05:00,Payment Data Systems Announces Apple Pay Suppo...,"san antonio, feb. 19, 2016 globe newswire -- p...",AAPL
4,2016-03-07,22:00:00,NTT DOCOMO Rolls Out FIDO Biometric Authentica...,"mountain view, ca-- marketwired - mar 7, 2016 ...",AAPL


**tokenization for cleaning punctuations and stop words**

In [ ]:

# import nltk
# from nltk.corpus import stopwords
# from nltk.tokenize import word_tokenize
# import string

# # Download required NLTK resources
# nltk.download('punkt')
# nltk.download('stopwords')

# # Prepare stopwords and punctuation sets
# stop_words = set(stopwords.words('english'))
# punctuations = set(string.punctuation)

# # Function to tokenize and clean text
# def tokenize_and_clean(text):
#     # 1. Word-level tokenization
#     tokens = word_tokenize(text)
#     # 2. Remove stopwords and punctuation
#     tokens = [t for t in tokens if t.lower() not in stop_words and t not in punctuations]
#     return tokens

# # Apply to your dataframe
# AAPL_News_tokenized['content'] = AAPL_News_tokenized['content'].apply(tokenize_and_clean)

# # Preview results
# AAPL_News_tokenized.head()


[nltk_data] Downloading package punkt to /usr/share/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /usr/share/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


,date,time,title,content,symbols
0,2012-05-04,00:00:00,Why Erie Indemnity's Shares Dropped,"[meaningful, another, movement]",AAPL
1,2015-09-25,04:01:00,"Security Solutions by AMAG, Identiv, ImageWare...","[new, york, ny, --, marketwired, sep, 25, 2015...",AAPL
2,2015-11-03,15:00:00,FIDO Alliance Announces 72 Certified Authentic...,"[mountain, view, ca, --, marketwired, nov, 3, ...",AAPL
3,2016-02-19,15:05:00,Payment Data Systems Announces Apple Pay Suppo...,"[san, antonio, feb., 19, 2016, globe, newswire...",AAPL
4,2016-03-07,22:00:00,NTT DOCOMO Rolls Out FIDO Biometric Authentica...,"[mountain, view, ca, --, marketwired, mar, 7, ...",AAPL


In [ ]:
# # 1. Get length of each list of tokens
# token_counts = AAPL_News_tokenized['content'].apply(len)

# # 2. Find the maximum and the sum
# max_length = token_counts.max()
# total_tokens = token_counts.sum()

# print(f"Longest news tokens: {max_length}")
# print(f"Total tokens in dataset: {total_tokens}")

Longest news tokens: 7802
Total tokens in dataset: 9962085


In [ ]:
AAPL_News_new = AAPL_News_tokenized.copy()
AAPL_News_new.head()

,date,time,title,content,symbols
0,2012-05-04,00:00:00,Why Erie Indemnity's Shares Dropped,is this meaningful? or just another movement?,AAPL
1,2015-09-25,04:01:00,"Security Solutions by AMAG, Identiv, ImageWare...","new york, ny-- marketwired - sep 25, 2015 - id...",AAPL
2,2015-11-03,15:00:00,FIDO Alliance Announces 72 Certified Authentic...,"mountain view, ca-- marketwired - nov 3, 2015 ...",AAPL
3,2016-02-19,15:05:00,Payment Data Systems Announces Apple Pay Suppo...,"san antonio, feb. 19, 2016 globe newswire -- p...",AAPL
4,2016-03-07,22:00:00,NTT DOCOMO Rolls Out FIDO Biometric Authentica...,"mountain view, ca-- marketwired - mar 7, 2016 ...",AAPL


In [ ]:
AAPL_News_new = AAPL_News_new[['content']]
AAPL_News_new.head()

,content
0,is this meaningful? or just another movement?
1,"new york, ny-- marketwired - sep 25, 2015 - id..."
2,"mountain view, ca-- marketwired - nov 3, 2015 ..."
3,"san antonio, feb. 19, 2016 globe newswire -- p..."
4,"mountain view, ca-- marketwired - mar 7, 2016 ..."


In [ ]:
AAPL_News_new.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19005 entries, 0 to 19004
Data columns (total 1 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   content  19005 non-null  object
dtypes: object(1)
memory usage: 148.6+ KB


In [ ]:
AAPL_News_sample = AAPL_News_new.sample(n=500, random_state=42)
AAPL_News_sample.head()

,content
1911,before the billions the first jobs of some of ...
17165,"dublin, aug. 19, 2025 globe newswire -- the me..."
2929,infosys limited infy is scheduled to report fo...
2968,fidelity strategic dividend income fund fsdix ...
10407,coinbase coin shares of coinbase coin surged m...


In [ ]:
AAPL_News_sample.info()

<class 'pandas.core.frame.DataFrame'>
Index: 500 entries, 1911 to 12490
Data columns (total 1 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   content  500 non-null    object
dtypes: object(1)
memory usage: 7.8+ KB


In [ ]:
import torch , gc
import pandas as pd
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.nn.functional import softmax

# 1. Setup GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [ ]:
# #Load Model/Tokenizer
# model_name = "soleimanian/financial-roberta-large-sentiment"
# tokenizer = AutoTokenizer.from_pretrained(model_name)
# model = AutoModelForSequenceClassification.from_pretrained(model_name)
# model = torch.nn.DataParallel(model)    #Using both GPU
# model.to(device)  # Move model to GPU
# model.eval()


# # 1. Clear everything before starting
# gc.collect()
# torch.cuda.empty_cache()

# def analyze_long_sentiment_safe(text):
#     try:
#         # Standard sliding window tokenization
#         inputs = tokenizer(text, return_tensors="pt", truncation=True,
#                            padding=True, max_length=512,
#                            stride=128, return_overflowing_tokens=True)

#         inputs = {k: v.to(device) for k, v in inputs.items()}  # move all tensors to GPU

#         inputs.pop("overflow_to_sample_mapping", None)

#         with torch.no_grad():
#             outputs = model(**inputs)
#             probs = softmax(outputs.logits, dim=-1).cpu().numpy()

#         avg_probs = np.mean(probs, axis=0)

#         # Clean up variables immediately
#         del inputs, outputs, probs
#         gc.collect()
#         torch.cuda.empty_cache()

#         return {
#             "negative": round(float(avg_probs[0]), 3),
#             "neutral": round(float(avg_probs[1]), 3),
#             "positive": round(float(avg_probs[2]), 3),
#             "sentiment": ["negative", "neutral", "positive"][np.argmax(avg_probs)]
#         }
#     except Exception as e:
#         return {"negative": 0, "neutral": 1, "positive": 0, "sentiment": "failed"}

# # 2. Run in a loop to allow periodic memory clearing
# final_results = []
# for i, text in enumerate(AAPL_News_sample['content']):
#     final_results.append(analyze_long_sentiment_safe(text))

#     # EVERY 10 ARTICLES: Force a deep memory clean
#     if i % 10 == 0:
#         gc.collect()
#         torch.cuda.empty_cache()

# # 3. Save results
# results_df = pd.DataFrame(final_results)
# # Match the index of the results to the original sample's index
# results_df.index = AAPL_News_sample.index
# AAPL_News_sample = pd.concat([AAPL_News_sample, results_df], axis=1)

In [ ]:
AAPL_News_sample.head()

,content
1911,before the billions the first jobs of some of ...
17165,"dublin, aug. 19, 2025 globe newswire -- the me..."
2929,infosys limited infy is scheduled to report fo...
2968,fidelity strategic dividend income fund fsdix ...
10407,coinbase coin shares of coinbase coin surged m...
...,...
6505,companies in many different sectors are starti...
11506,voip-pal.com inc. ceo emil malak discusses voi...
7700,if artificial intelligence ai was the story of...
16368,company logo explore comprehensive insights in...


In [ ]:
# # This shows the total count for positive, neutral, negative, and failed
# print(AAPL_News_sample['sentiment'].value_counts())

sentiment
positive    509
negative    272
neutral     219
Name: count, dtype: int64


**running on all data**

In [ ]:
#Load Model/Tokenizer
model_name = "soleimanian/financial-roberta-large-sentiment"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)
model = torch.nn.DataParallel(model)    #Using both GPU
model.to(device)  # Move model to GPU
model.eval()


# 1. Clear everything before starting
gc.collect()
torch.cuda.empty_cache




def analyze_long_sentiment_safe(text):
    try:
        inputs = tokenizer(
            text,
            return_tensors="pt",
            truncation=True,
            padding=True,
            max_length=512,
            stride=256,
            return_overflowing_tokens=True
        )

        # REMOVE mapping before moving tensors to GPU
        inputs.pop("overflow_to_sample_mapping")

        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            logits = model(**inputs).logits
            probs = softmax(logits, dim=-1).cpu().numpy()

        avg_probs = probs.mean(axis=0)

        return {
            "negative": round(float(avg_probs[0]), 3),
            "neutral": round(float(avg_probs[1]), 3),
            "positive": round(float(avg_probs[2]), 3),
            "sentiment": ["negative", "neutral", "positive"][avg_probs.argmax()]
        }

    except Exception:
        return {"negative": 0, "neutral": 1, "positive": 0, "sentiment": "failed"}




# 2. Run in a loop to allow periodic memory clearing
final_results = []
for i, text in enumerate(AAPL_News_new['content']):
    final_results.append(analyze_long_sentiment_safe(text))

    # EVERY 10 ARTICLES: Force a deep memory clean
    if i % 10 == 0:
        gc.collect()
        torch.cuda.empty_cache()

# 3. Save results
results_df = pd.DataFrame(final_results)
# Match the index of the results to the original  index
results_df.index = AAPL_News_new.index


In [ ]:
AAPL_News_new.head()

,content
0,is this meaningful? or just another movement?
1,"new york, ny-- marketwired - sep 25, 2015 - id..."
2,"mountain view, ca-- marketwired - nov 3, 2015 ..."
3,"san antonio, feb. 19, 2016 globe newswire -- p..."
4,"mountain view, ca-- marketwired - mar 7, 2016 ..."


In [ ]:
results_df.head()

,negative,neutral,positive,sentiment
0,0.0,0.999,0.000,neutral
1,0.0,0.399,0.601,positive
2,0.0,0.194,0.805,positive
3,0.0,0.339,0.661,positive
4,0.0,0.324,0.676,positive


In [ ]:
print(results_df['sentiment'].value_counts())

sentiment
positive    9909
negative    5109
neutral     3987
Name: count, dtype: int64


In [ ]:
# Combine 'content' with results into a new DataFrame
AAPL_News_results = pd.concat(
    [AAPL_News_new['content'], results_df[['sentiment', 'negative', 'neutral', 'positive']]],
    axis=1
)
AAPL_News_results.head()

,content,sentiment,negative,neutral,positive
0,is this meaningful? or just another movement?,neutral,0.0,0.999,0.000
1,"new york, ny-- marketwired - sep 25, 2015 - id...",positive,0.0,0.399,0.601
2,"mountain view, ca-- marketwired - nov 3, 2015 ...",positive,0.0,0.194,0.805
3,"san antonio, feb. 19, 2016 globe newswire -- p...",positive,0.0,0.339,0.661
4,"mountain view, ca-- marketwired - mar 7, 2016 ...",positive,0.0,0.324,0.676


In [ ]:
AAPL_News_results.tail()

,content,sentiment,negative,neutral,positive
19000,key points alphabet is growing at a market-bea...,positive,0.000,0.002,0.997
19001,"every may, tens of thousands of investors desc...",neutral,0.000,0.515,0.485
19002,key points buffett's firm berkshire hathaway h...,positive,0.000,0.013,0.987
19003,"new york ap prices for the nasdaq composite, n...",neutral,0.000,0.999,0.000
19004,explore the s p500 index on friday and find ou...,negative,0.998,0.001,0.000


In [ ]:
AAPL_News_results.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19005 entries, 0 to 19004
Data columns (total 5 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   content    19005 non-null  object 
 1   sentiment  19005 non-null  object 
 2   negative   19005 non-null  float64
 3   neutral    19005 non-null  float64
 4   positive   19005 non-null  float64
dtypes: float64(3), object(2)
memory usage: 742.5+ KB


In [ ]:
# Merge tokenized data and results
AAPL_RoBERTA_Results = pd.concat([AAPL_News_tokenized, AAPL_News_results], axis=1)

# Remove duplicated columns
AAPL_RoBERTA_Results = AAPL_RoBERTA_Results.loc[:, ~AAPL_RoBERTA_Results.columns.duplicated()]

# Ensure index matches the tokenized data
AAPL_RoBERTA_Results.index = AAPL_News_tokenized.index

# View first few rows
AAPL_RoBERTA_Results.head()


,date,time,title,content,symbols,sentiment,negative,neutral,positive
0,2012-05-04,00:00:00,Why Erie Indemnity's Shares Dropped,is this meaningful? or just another movement?,AAPL,neutral,0.0,0.999,0.000
1,2015-09-25,04:01:00,"Security Solutions by AMAG, Identiv, ImageWare...","new york, ny-- marketwired - sep 25, 2015 - id...",AAPL,positive,0.0,0.399,0.601
2,2015-11-03,15:00:00,FIDO Alliance Announces 72 Certified Authentic...,"mountain view, ca-- marketwired - nov 3, 2015 ...",AAPL,positive,0.0,0.194,0.805
3,2016-02-19,15:05:00,Payment Data Systems Announces Apple Pay Suppo...,"san antonio, feb. 19, 2016 globe newswire -- p...",AAPL,positive,0.0,0.339,0.661
4,2016-03-07,22:00:00,NTT DOCOMO Rolls Out FIDO Biometric Authentica...,"mountain view, ca-- marketwired - mar 7, 2016 ...",AAPL,positive,0.0,0.324,0.676


In [ ]:
AAPL_RoBERTA_Results.tail()


,date,time,title,content,symbols,sentiment,negative,neutral,positive
19000,2025-08-29,09:00:00,Artifical Intelligence (AI) Is on Sale: 2 Stoc...,key points alphabet is growing at a market-bea...,AAPL,positive,0.000,0.002,0.997
19001,2025-08-29,10:22:00,Here Are Billionaire Warren Buffett's 5 Bigges...,"every may, tens of thousands of investors desc...",AAPL,neutral,0.000,0.515,0.485
19002,2025-08-29,14:15:00,2 Top Buffett Stocks to Buy and Hold for the L...,key points buffett's firm berkshire hathaway h...,AAPL,positive,0.000,0.013,0.987
19003,2025-08-29,14:30:18,BC-Most Active Stocks,"new york ap prices for the nasdaq composite, n...",AAPL,neutral,0.000,0.999,0.000
19004,2025-08-29,18:05:03,Which S&amp;P500 stocks are the most active on...,explore the s p500 index on friday and find ou...,AAPL,negative,0.998,0.001,0.000


In [ ]:
# 1. Save as parquet
file_path = "/kaggle/working/AAPL_RoBERTA_Results.parquet"
AAPL_RoBERTA_Results.to_parquet(file_path, index=True)

# 2. The file will appear in Kaggle's "Output" tab and can be downloaded from there
print(f"Saved to {file_path}")


Saved to /kaggle/working/AAPL_RoBERTA_Results.parquet


In [ ]:
merged = AAPL_News_tokenized.join(output.drop(columns=['content']), how='left')
merged.head()

,date,time,title,content,symbols,negative,neutral,positive,sentiment
0,2012-05-04,00:00:00,Why Erie Indemnity's Shares Dropped,is this meaningful? or just another movement?,AAPL,0.001,0.001,0.998,positive
1,2015-09-25,04:01:00,"Security Solutions by AMAG, Identiv, ImageWare...","new york, ny-- marketwired - sep 25, 2015 - id...",AAPL,0.001,0.001,0.998,positive
2,2015-11-03,15:00:00,FIDO Alliance Announces 72 Certified Authentic...,"mountain view, ca-- marketwired - nov 3, 2015 ...",AAPL,0.256,0.245,0.499,positive
3,2016-02-19,15:05:00,Payment Data Systems Announces Apple Pay Suppo...,"san antonio, feb. 19, 2016 globe newswire -- p...",AAPL,0.002,0.202,0.796,positive
4,2016-03-07,22:00:00,NTT DOCOMO Rolls Out FIDO Biometric Authentica...,"mountain view, ca-- marketwired - mar 7, 2016 ...",AAPL,0.005,0.050,0.945,positive


In [ ]:
 print(merged['sentiment'].value_counts())

sentiment
positive    9909
negative    5109
neutral     3987
Name: count, dtype: int64


In [ ]:
merged.to_parquet("/kaggle/working/AAPL_RoBERTA_finalresults.parquet", index=True)


In [ ]:
import shutil
shutil.move("/kaggle/working/AAPL_RoBERTA_finalresults.parquet", "/kaggle/temp/AAPL_RoBERTA_finalresults.parquet")


FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/temp/AAPL_RoBERTA_finalresults.parquet'